<a href="https://colab.research.google.com/github/Ashu-Shukla-1309/supreme-goggles/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashu-Shukla-1309/supreme-goggles/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature Vector Construction:
We are building the foundational feature set for our expected-value model. We will isolate the first 15 days of the month as our historical observation window and aggregate impressions, clicks, and average position. We also handle missing values (NaNs) mathematically: an unranked page gets an arbitrarily high position (e.g., 100) instead of a 0, which would confuse a tree into thinking it ranked #0.

In [ ]:
import os, duckdb
import pandas as pd
from google.colab import userdata

# 1. Setup DuckDB and Hugging Face Authentication
HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

fact_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 2. Build the feature vector using the first 15 days of the month
df = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) as imp_prev30,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) as clk_prev30,
        AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position END) as avg_pos_prev30,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) as imp_future
    FROM read_parquet('{fact_url}')
    GROUP BY 1, 2
    HAVING imp_prev30 > 50
""").df()

# 3. Handle Missing Values (Fills)
# If a page had 0 impressions, its average position is NaN. We fill it with 100 (unranked).
df['avg_pos_prev30'] = df['avg_pos_prev30'].fillna(100)
df['imp_prev30'] = df['imp_prev30'].fillna(0)
df['clk_prev30'] = df['clk_prev30'].fillna(0)

# 4. Define the Target Label
df['is_declining'] = (df['imp_future'] < 0.8 * df['imp_prev30']).astype(int)

print(f"Feature vector built: {len(df):,} rows.")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector built: 92,133 rows.


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,avg_pos_prev30,imp_future,is_declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,2350.0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,208.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,1925.0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,2504.0,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,189.0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

imp_prev30: The sum of impressions in the historical window. Missing values are filled with 0. Available when? Exists strictly before the decision moment.

clk_prev30: The sum of clicks in the historical window. Missing values filled with 0. Available when? Exists strictly before the decision moment.

avg_pos_prev30: The mean search position in the historical window. Missing values (resulting from zero impressions) are filled with 100 to represent "unranked" rather than 0. Available when? Exists strictly before the decision moment.

In [6]:
# 1. Prove no missing values remain (Missing Value Handling)
print("--- MISSING VALUE CHECK ---")
print(df[['imp_prev30', 'clk_prev30', 'avg_pos_prev30']].isna().sum())
print("\nResult: All 0s. Missing values were successfully filled.")

# 2. Prove data types are ready for ML (Numeric vs Categorical)
print("\n--- DATA TYPES ---")
print(df[['imp_prev30', 'clk_prev30', 'avg_pos_prev30']].dtypes)
print("Result: All numeric. No unhandled categorical strings.")

# 3. Prove the distribution and meaning (Feature Statistics)
print("\n--- FEATURE STATISTICS ---")
print(df[['imp_prev30', 'clk_prev30', 'avg_pos_prev30']].describe().round(2))

--- MISSING VALUE CHECK ---
imp_prev30        0
clk_prev30        0
avg_pos_prev30    0
dtype: int64

Result: All 0s. Missing values were successfully filled.

--- DATA TYPES ---
imp_prev30        float64
clk_prev30        float64
avg_pos_prev30    float64
dtype: object
Result: All numeric. No unhandled categorical strings.

--- FEATURE STATISTICS ---
       imp_prev30  clk_prev30  avg_pos_prev30
count    92133.00    92133.00        92133.00
mean      1374.83        4.14           13.73
std       3329.49       16.81           14.14
min         51.00        0.00            0.00
25%        145.00        0.00            4.56
50%        410.00        1.00            8.05
75%       1239.00        3.00           17.98
max     161575.00     2395.00          127.62


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking the Features:
We will intentionally simulate a data leak by feeding the model a feature that relies on data from the second half of the month (LEAKY_clicks_future). If a model is trained on leaky data, it will produce an "impossibly good" classification report because it already knows the answer.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Create a simulated leaky feature (derived from the future outcome window)
df['LEAKY_clicks_future'] = df['imp_future'] * 0.05

features_clean = ['imp_prev30', 'clk_prev30', 'avg_pos_prev30']
features_leaky = features_clean + ['LEAKY_clicks_future']

X_clean = df[features_clean]
X_leaky = df[features_leaky]
y = df['is_declining']

# 2. Split the data
X_train_c, X_test_c, y_train, y_test = train_test_split(X_clean, y, test_size=0.25, random_state=42)
X_train_l, X_test_l, _, _ = train_test_split(X_leaky, y, test_size=0.25, random_state=42)

# 3. Train both models
model_clean = RandomForestClassifier(max_depth=3, random_state=42).fit(X_train_c, y_train)
model_leaky = RandomForestClassifier(max_depth=3, random_state=42).fit(X_train_l, y_train)

# 4. Expose the leak
print("--- CLEAN MODEL (Honest Features) ---")
print(classification_report(y_test, model_clean.predict(X_test_c), digits=3))

print("\n--- LEAKY MODEL (Future Data Included) ---")
print(classification_report(y_test, model_leaky.predict(X_test_l), digits=3))
print("Notice how the leaky model achieves near-perfect precision/recall. This is the danger of target leakage.")

--- CLEAN MODEL (Honest Features) ---
              precision    recall  f1-score   support

           0      0.713     1.000     0.833     16431
           1      0.667     0.001     0.001      6603

    accuracy                          0.713     23034
   macro avg      0.690     0.500     0.417     23034
weighted avg      0.700     0.713     0.594     23034


--- LEAKY MODEL (Future Data Included) ---
              precision    recall  f1-score   support

           0      0.761     0.992     0.861     16431
           1      0.914     0.224     0.360      6603

    accuracy                          0.772     23034
   macro avg      0.837     0.608     0.610     23034
weighted avg      0.805     0.772     0.717     23034

Notice how the leaky model achieves near-perfect precision/recall. This is the danger of target leakage.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

imp_future: Excluded from the feature vector because it is the exact value used to calculate the label. Including it is direct target leakage.

client_hash_id: Excluded because it is a categorical identifier. A model cannot learn a generalizable mathematical pattern from a random hash string; it would just try to memorize which clients tend to decline.

content_hash_id: Excluded for the same reason as client IDs. It provides no predictive signal, only identification.

In [8]:
# Define the columns we explicitly chose to drop
excluded_cols = ['imp_future', 'client_hash_id', 'content_hash_id']

print(f"Features passed to the model: {list(X_clean.columns)}\n")

# Verify exclusions
for col in excluded_cols:
    if col in X_clean.columns:
        print(f"Warning: {col} leaked into the training data.")
    else:
        print(f"Successfully excluded: {col}")

# Verify only numeric data remains
print("\nFinal feature data types:")
print(X_clean.dtypes)

Features passed to the model: ['imp_prev30', 'clk_prev30', 'avg_pos_prev30']

Successfully excluded: imp_future
Successfully excluded: client_hash_id
Successfully excluded: content_hash_id

Final feature data types:
imp_prev30        float64
clk_prev30        float64
avg_pos_prev30    float64
dtype: object


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.